In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

metadata_df = metadata_df.rename(columns={'patient id': 'Patient ID'})

metadata_df

,sample_title,Patient ID,disease state,tissue
samples,,,,
GSM5574433,Patient ID: B012-0001 [SL462108],B012-0001,esophageal and gastroesophageal junction adeno...,tumor
GSM5574434,Patient ID: B012-0002 [SL463401],B012-0002,esophageal and gastroesophageal junction adeno...,tumor
GSM5574435,Patient ID: B012-0003 [SL446517],B012-0003,esophageal and gastroesophageal junction adeno...,tumor
GSM5574436,Patient ID: B012-0004 [SL462135],B012-0004,esophageal and gastroesophageal junction adeno...,tumor
GSM5574437,Patient ID: B012-0005 [SL446533],B012-0005,esophageal and gastroesophageal junction adeno...,tumor
GSM5574438,Patient ID: B012-0006 [SL446518],B012-0006,esophageal and gastroesophageal junction adeno...,tumor
GSM5574439,Patient ID: B012-0007 [SL446519],B012-0007,esophageal and gastroesophageal junction adeno...,tumor
GSM5574440,Patient ID: B012-0008 [SL462109],B012-0008,esophageal and gastroesophageal junction adeno...,tumor
GSM5574441,Patient ID: B012-0011 [SL446521],B012-0011,esophageal and gastroesophageal junction adeno...,tumor


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()

# https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE183924&format=file&file=GSE183924%5FClinical%5FData%5Fwith%5FCIBERSORT%5Fanalysis%5Ffor%5FGEO%2Exlsx
df_clinical_data = pd.read_excel('GSE183924_Clinical_Data_with_CIBERSORT_analysis_for_GEO.xlsx', sheet_name=0)
df_clinical_data.to_csv('GSE183924_Clinical_Data_with_CIBERSORT_analysis_for_GEO.csv', index=False)

df_combined = df_clinical_data.merge(
    df_combined,
    left_on='Patient ID',
    right_on='Patient ID',
    how='inner'
)


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.to_csv(data_file_name, index=False)
df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_1128/618446365.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,Patient ID,RNA-Seq ID,Date of Esophageal Surgery,Relapse,Date of Relapse or Censoring,Date of Death,RFS Censor,RFS (months),B cells naive,B cells memory,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
0,B012-0001,SL462108,2016-04-01,No,2018-06-04,NaT,0,26.1184,0.022107,0.000000,...,7070,9,10,10,4486,1480,122,2884,5,46
1,B012-0004,SL462135,2016-06-10,No,2018-06-27,NaT,0,24.5724,0.019443,0.000000,...,1740,12,18,48,1647,576,70,885,4,15
2,B012-0007,SL446519,2016-08-01,Yes,2017-10-02,NaT,1,14.0461,0.037155,0.000000,...,16814,18,27,36,10114,3915,364,7216,7,63
3,B012-0008,SL462109,2016-10-14,No,2018-03-19,NaT,0,17.1382,0.010599,0.000000,...,3426,4,9,12,2312,799,60,1439,11,44
4,B012-0011,SL446521,2016-12-19,No,2019-01-17,2019-01-17,1,24.9671,0.030652,0.002470,...,7066,10,17,29,5503,2412,236,2636,9,49
5,B012-0012,SL462146,2016-12-23,No,2020-05-03,2020-05-03,1,40.3618,0.039712,0.000000,...,6919,7,3,5,4560,1455,130,3822,2,58
6,B012-0013,SL446522,2017-01-04,Yes,2017-04-10,2018-03-13,1,3.1579,0.014226,0.000000,...,6854,7,18,29,5298,2259,293,3148,7,46
7,B012-0015,SL446523,2017-02-10,No,2018-09-17,NaT,0,19.2105,0.012786,0.049479,...,19902,17,18,20,17562,6599,542,7642,8,56
8,B012-0017,SL446534,2017-02-16,Yes,2019-03-27,NaT,1,25.2961,0.017473,0.000000,...,20250,20,21,18,14568,6538,538,8608,11,87
9,B012-0018,SL462114,2017-03-08,Yes,2017-11-09,2018-10-20,1,8.0921,0.012929,0.001770,...,414,0,0,2,325,134,22,217,2,0


# Generate description

In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, 'GSE183924_Clinical_Data_with_CIBERSORT_analysis_for_GEO.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': df_combined.shape[1] - df_raw_counts.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
